In [1]:
import os
os.chdir("..")
import numpy as np
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

import gzip
from utils.config import get_co2_levels, path_to_clustering_results_sclopf
from utils.clustering_visualisation import *
from utils import data_handling
from utils.config import path_to_pypsa_network_sclopf, path_to_figures_sclopf
from utils import cascade_simulation
from utils.clustering import load_clustering

# Load network graph and node positions, the same for all CO2 levels
snet_index = 0
n_nodes = 600
network = data_handling.load_pypsa_network(0.6, n_nodes, True)
nx_graph = data_handling.build_networkx_graph(network, snet_index= snet_index)
pos = nx.get_node_attributes(nx_graph, 'pos')
I_m, B_d, num_parallels, line_limits = data_handling.get_matrices_from_nx_graph(nx_graph)

# Determine number of simulations,
# i.e., total number of initial failures over the simulated period of one year
bridge_idxs = data_handling.nx_edges_to_matrix_indices(nx.bridges(nx_graph), nx_graph)
double_line_failures = cascade_simulation.calc_possible_double_line_failures(num_parallels, ignored_idxs=bridge_idxs)
num_snapshots_weighted = network.snapshot_weightings.objective.sum()
num_failures_weighted = len(double_line_failures) * num_snapshots_weighted

n_nodes_split, lost_load_share = None, 0.05
co2l_list = get_co2_levels(n_nodes=n_nodes,)
n_clusters = 256
indicator_type = "rocof"
transformation = "blackout"
clustering_algorithm = "optics"
max_dist = 3
decay = 1


# save_dir = path_to_clustering_results

save_dir = get_path_to_clustering_dir(
        n_nodes=n_nodes,
        co2l="all",
        indicator_type=indicator_type,
        transformation=transformation,
        n_nodes_split=n_nodes_split,
        lost_load_share=lost_load_share,
    )
path_to_clustering_results = save_dir

/srv/data/jlange/power-system-split/no_extensions//data/European_networks_sclopf/sclopf-elec_s_600_ec_lv1.0_Co2L0.6-2920SEG.nc


/home/mtitz/.conda/envs/system_split/lib/python3.11/site-packages/pypsa/components.py:323: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  attrs.loc[bool_b, "default"] = attrs.loc[bool_b].isin({True, "True"})
/home/mtitz/.conda/envs/system_split/lib/python3.11/site-packages/pypsa/components.py:323: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  attrs.loc[bool_b, "default"] = attrs.loc[bool_b].isin({True, "True"})
/home/mtitz/.conda/envs/system_split/lib/python3.11/site-packages/pypsa/components.py:323: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dt

In [2]:
co2l = 0.05
component_props_level = pd.read_hdf(
            path_to_evaluation_results_sclopf
            + f"component_properties_Co2L{co2l}_n{n_nodes}.h5"
        )
component_props_level["total_split_number"] = ((((((component_props_level[(component_props_level.split_number < component_props_level.split_number.shift(1).fillna(0).astype(int)).shift(-1).fillna(False)].split_number + 1))))).reindex(
    component_props_level.index, fill_value=0
).cumsum().fillna(0).astype(int).shift(1)+ component_props_level.split_number).fillna(0).astype(int)

/tmp/ipykernel_750637/3438404727.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  component_props_level["total_split_number"] = ((((((component_props_level[(component_props_level.split_number < component_props_level.split_number.shift(1).fillna(0).astype(int)).shift(-1).fillna(False)].split_number + 1))))).reindex(


In [5]:
component_props_level.power_imbalance*(component_props_level.power_imbalance<0).astype(int)/ component_props_level.load

0         -0.016703
1          0.000000
2         -0.016703
3          0.000000
4         -0.000100
             ...   
7731777   -0.003125
7731778    0.000000
7731779   -0.003756
7731780    0.000000
7731781    0.000000
Length: 7731782, dtype: float64

In [6]:
component_props_level.shedding_load_loss_share

0          0.016365
1          0.000000
2          0.016365
3          0.000000
4          0.000100
             ...   
7731777    0.003123
7731778    0.000000
7731779    0.003747
7731780    0.000000
7731781    0.000000
Name: shedding_load_loss_share, Length: 7731782, dtype: float64

In [47]:
component_props_level["total_split_number"] = ((((((component_props_level[(component_props_level.split_number < component_props_level.split_number.shift(1).fillna(0).astype(int)).shift(-1).fillna(False)].split_number + 1))))).reindex(
    component_props_level.index, fill_value=0
).cumsum().fillna(0).astype(int).shift(1)+ component_props_level.split_number).fillna(0).astype(int)

/tmp/ipykernel_739280/3458832354.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  component_props_level["total_split_number"] = ((((((component_props_level[(component_props_level.split_number < component_props_level.split_number.shift(1).fillna(0).astype(int)).shift(-1).fillna(False)].split_number + 1))))).reindex(


In [48]:
from utils.config import path_to_vis_results_sclopf

split_properties = pd.read_hdf(path_to_vis_results_sclopf + f"split_properties_all_n{n_nodes}.h5", index_col=0)

In [49]:
split_properties_lvl = pd.read_csv(path_to_vis_results_sclopf + f"split_properties_Co2L{co2l}_n{n_nodes}.csv", index_col=[0,1,2])

In [63]:
co2l_list = get_co2_levels(n_nodes)
all_comp_props = {}
for co2l in co2l_list:
    
    component_props_level = pd.read_hdf(
                path_to_evaluation_results_sclopf
                + f"component_properties_Co2L{co2l}_n{n_nodes}.h5"
            )
    
    all_comp_props[co2l] = component_props_level

In [ ]:

for co2l in co2l_list:
    component_props_level = all_comp_props[co2l]
    component_props_level["total_split_number"] = ((((((component_props_level[(component_props_level.split_number < component_props_level.split_number.shift(1).fillna(0).astype(int)).shift(-1).fillna(False)].split_number + 1))))).reindex(
    component_props_level.index, fill_value=0
    ).cumsum().fillna(0).astype(int).shift(1)+ component_props_level.split_number).fillna(0).astype(int)
    print(f"CO2 Level: {co2l}")
    print(f"{all_comp_props[co2l].total_split_number.iloc[-1]}, {all_vecs[co2l].shape[0]}")

CO2 Level: 0.6
1799794, 1799795
CO2 Level: 0.5
2035385, 2035386
CO2 Level: 0.4
2334116, 2334117
CO2 Level: 0.3
2617501, 2617502
CO2 Level: 0.2
2801625, 2801626
CO2 Level: 0.1
2968728, 2968729
CO2 Level: 0.05
2968728, 2968729
CO2 Level: 0.0
4709979, 4709980


In [94]:
all_comp_props[0.05] == all_comp_props[0.1]

,time_stamp,init_failure_0,init_failure_1,split_number,rot_energy,power_imbalance,load,rocof,load_share,shedding_load_loss_share,blackout_load_loss_share,total_load_loss_share,snapshot_weighting,total_split_number
0,True,True,True,True,False,False,True,False,True,False,True,False,True,True
1,True,True,True,True,True,False,True,False,True,True,True,True,True,True
2,True,True,True,True,False,False,True,False,True,False,True,False,True,True
3,True,True,True,True,True,False,True,False,True,True,True,True,True,True
4,True,True,True,True,False,False,True,False,True,False,True,False,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7731777,True,True,True,True,False,False,True,False,True,False,True,False,True,True
7731778,True,True,True,True,True,False,True,False,True,True,True,True,True,True
7731779,True,True,True,True,False,False,True,False,True,False,True,False,True,True
7731780,True,True,True,True,True,False,True,False,True,True,True,True,True,True


In [78]:
co2l_list = get_co2_levels(n_nodes)
all_vecs = {}
for co2l in co2l_list:
    
    f = f'/srv/data/jlange/power-system-split/no_extensions//results/sclopf/evaluation_results//indicator_vector_rocof_Co2L{co2l}_n{n_nodes}.pklz'
    with gzip.open(f, "rb") as out:
        vectors_tuple = pickle.load(out)
    
    all_vecs[co2l] = vectors_tuple[-1]

In [98]:
import os

base_dir = "/srv/data/jlange/power-system-split/no_extensions"
for root, dirs, files in os.walk(base_dir):
    for fname in files:
        if "0.1" in fname and "600" in fname:
            fpath_01 = os.path.join(root, fname)
            fname_005 = fname.replace("0.1", "0.05")
            fpath_005 = os.path.join(root, fname_005)
            fname_00 = fname.replace("0.1", "0.0")
            fpath_00 = os.path.join(root, fname_00)
            if os.path.exists(fpath_005):
                size_01 = os.path.getsize(fpath_01)
                size_005 = os.path.getsize(fpath_005)
                print(f"{fname}: {size_01} bytes")
                print(f"{fname_005}: {size_005} bytes")
            if os.path.exists(fpath_00):
                size_00 = os.path.getsize(fpath_00)
                print(f"{fname_00}: {size_00} bytes\n")

sclopf-elec_s_600_ec_lv1.0_Co2L0.1-2920SEG.nc: 679853998 bytes
sclopf-elec_s_600_ec_lv1.0_Co2L0.05-2920SEG.nc: 679644686 bytes
sclopf-elec_s_600_ec_lv1.0_Co2L0.0-2920SEG.nc: 668940374 bytes

system_splits_Co2L0.1_n600.pklz: 19889390 bytes
system_splits_Co2L0.05_n600.pklz: 19889391 bytes
system_splits_Co2L0.0_n600.pklz: 35400512 bytes

component_indicator_vectors_Co2L0.1_n600.pklz: 90446851 bytes
component_indicator_vectors_Co2L0.05_n600.pklz: 90446852 bytes
component_indicator_vectors_Co2L0.0_n600.pklz: 158525425 bytes

component_properties_Co2L0.1_n600.h5: 865968872 bytes
component_properties_Co2L0.05_n600.h5: 865968872 bytes
component_properties_Co2L0.0_n600.h5: 1561581832 bytes

component_indicator_vectors_Co2L0.1_n600.pklz: 90446851 bytes
component_indicator_vectors_Co2L0.05_n600.pklz: 90446852 bytes
component_indicator_vectors_Co2L0.0_n600.pklz: 158525425 bytes

component_properties_Co2L0.1_n600.h5: 865968872 bytes
component_properties_Co2L0.05_n600.h5: 865968872 bytes
component_p

In [113]:
net1 = data_handling.load_pypsa_network(0.1, n_nodes, True)
net5 = data_handling.load_pypsa_network(0.05, n_nodes, True)

/srv/data/jlange/power-system-split/no_extensions//data/European_networks_sclopf/sclopf-elec_s_600_ec_lv1.0_Co2L0.1-2920SEG.nc


INFO:pypsa.io:Imported network sclopf-elec_s_600_ec_lv1.0_Co2L0.1-2920SEG.nc has buses, carriers, generators, global_constraints, lines, links, loads, storage_units


/srv/data/jlange/power-system-split/no_extensions//data/European_networks_sclopf/sclopf-elec_s_600_ec_lv1.0_Co2L0.05-2920SEG.nc


INFO:pypsa.io:Imported network sclopf-elec_s_600_ec_lv1.0_Co2L0.05-2920SEG.nc has buses, carriers, generators, global_constraints, lines, links, loads, storage_units


In [ ]:
net1.

False

In [109]:
from utils.config import path_to_cascade_results_sclopf

cascade_dicts = {}
for co2lvl in [0.1, 0.05]:

    path_to_cascade_results_file = (
                path_to_cascade_results_sclopf
                + f"system_splits_Co2L{co2lvl}_n{n_nodes}.pklz"
            )
    with gzip.open(path_to_cascade_results_file, "rb") as fh:
        cascade_dict = pickle.load(fh)
    cascade_dicts[co2lvl] = cascade_dict

In [ ]:
[len(cascade_dict.items([0])) for cascade_dict in cascade_dicts.values()]

[2920, 2920]

In [ ]:
list((list(cascade_dicts[0.0].items())[0][1].values()))[10]

In [112]:
cascade_dicts[0.1]== cascade_dicts[0.05]

True

In [ ]:
list((list(cascade_dicts[0.1].items())[0][1].values()))[110], list((list(cascade_dicts[0.05].items())[0][1].values()))[110]

([40, 108, 103, 233, 319, 73, 122, 133, 314, 326],
 [40, 108, 103, 233, 319, 73, 122, 133, 314, 326])

In [84]:
[vec.shape for vec in all_comp_props.values()]

[(4565254, 13),
 (5129777, 13),
 (5831365, 13),
 (6680884, 13),
 (7265240, 13),
 (7731782, 13),
 (7731782, 13),
 (13942612, 13)]

In [83]:
[vec.shape for vec in all_vecs.values()]

[(1799795, 456),
 (2035386, 456),
 (2334117, 456),
 (2617502, 456),
 (2801626, 456),
 (2968729, 456),
 (2968729, 456),
 (4709980, 456)]

In [ ]:
    if use_sclopf:
        path_to_cascade_results_file = (
            path_to_cascade_results
            + "/system_splits_Co2L"
            + f"{co2_lvl}_n{n_nodes}.pklz"
        )
    else:
        path_to_cascade_results_file = (
            path_to_cascade_results
            + "/system_splits_singlelinefailures_Co2L"
            + f"{co2_lvl}_n{n_nodes}_lopf.pklz"
        )

In [77]:
prop_first_comp = pd.concat([comp.iloc[0,:] for comp in all_comp_props.values()], axis=1).reset_index(drop=True).T
prop_first_comp.index = list(all_comp_props.keys())
prop_first_comp.columns = list(all_comp_props.values())[0].columns
prop_first_comp


,time_stamp,init_failure_0,init_failure_1,split_number,rot_energy,power_imbalance,load,rocof,load_share,shedding_load_loss_share,blackout_load_loss_share,total_load_loss_share,snapshot_weighting
0.60,2013-01-01 00:00:00,0,68,0,860433.730367,79.301252,220883.090045,0.002304,0.997123,0.0,0.0,0.0,3.0
0.50,2013-01-01 00:00:00,0,68,0,806030.313537,176.393493,221310.189487,0.005471,0.999051,0.0,0.0,0.0,3.0
0.40,2013-01-01 00:00:00,9,10,0,36186.9377,-5168.146247,8823.3059,-3.57045,0.039831,0.02333,0.039831,0.039831,3.0
0.30,2013-01-01 00:00:00,1,187,0,750269.373437,-12427.616025,214930.129783,-0.414105,0.970249,0.056101,0.0,0.056101,3.0
0.20,2013-01-01 00:00:00,0,730,0,748875.984237,-22.129413,221396.463401,-0.000739,0.99944,0.0001,0.0,0.0001,3.0
0.10,2013-01-01 00:00:00,0,7,0,629720.571287,-2875.309083,217033.834257,-0.11415,0.979746,0.01298,0.0,0.01298,3.0
0.05,2013-01-01 00:00:00,0,7,0,553983.654291,-3625.143999,217033.834257,-0.163594,0.979746,0.016365,0.0,0.016365,3.0
0.00,2013-01-01 00:00:00,0,2,0,548266.964036,4831.886379,215883.500924,0.220325,0.974553,0.0,0.0,0.0,3.0


In [ ]:
co2l_list = get_co2_levels(n_nodes)
for co2l in co2l_list:
    
    component_props_level = pd.read_hdf(
                path_to_evaluation_results_sclopf
                + f"component_properties_Co2L{co2l}_n{n_nodes}.h5"
            )
    
    f = f'/srv/data/jlange/power-system-split/no_extensions//results/sclopf/evaluation_results//indicator_vector_rocof_Co2L{co2l}_n{n_nodes}.pklz'
    with gzip.open(f, "rb") as out:
        vectors_tuple = pickle.load(out)

    is_same = component_props_level.rocof.max() == vectors_tuple[1].max().max()
    is_same = is_same and component_props_level.rocof.min() == vectors_tuple[1].min().min()
    
    print(f"co2l: {co2l}, is_same: {is_same}")


co2l: 0.6, is_same: True
co2l: 0.5, is_same: True
co2l: 0.4, is_same: True
co2l: 0.3, is_same: True
co2l: 0.2, is_same: True
co2l: 0.1, is_same: True
co2l: 0.05, is_same: False
co2l: 0.0, is_same: True


In [ ]:
+ f"{co2_lvl}_n{n_nodes}.pklz"
+ f"{co2_lvl}_n{n_nodes}_lopf.pklz"

In [52]:
# split_groups = component_props_level.groupby(["time_stamp", "split_number"])
# split_props_ = pd.DataFrame(
#     index=split_groups.groups.keys(),
# )

In [ ]:
f = f'/srv/data/jlange/power-system-split/no_extensions//results/sclopf/evaluation_results//indicator_vector_rocof_Co2L{co2l}_n{n_nodes}.pklz'

with gzip.open(f, "rb") as out:
    vectors_tuple = pickle.load(out)
    

In [25]:
split_comp = component_props_level[(component_props_level.time_stamp==pd.Timestamp("2013-01-01 09:00:00")) & (component_props_level.split_number==351)]

In [55]:
component_props_level.rocof

0           2.203254e-01
1          -1.236084e+01
2           4.562836e-01
3          -1.236084e+01
4          -6.639400e+00
                ...     
13942607   -1.166595e+11
13942608    2.893492e-03
13942609   -1.166595e+11
13942610   -1.611931e-01
13942611    6.498965e+12
Name: rocof, Length: 13942612, dtype: float64

In [56]:
[np.unique(vectors_tuple[-1][i]) for i in range(10)]

[array([-12.3608385 ,   0.22032544]),
 array([-1.23608385e+01, -6.63940048e+00,  4.56283600e-01,  2.96343914e+12]),
 array([-9.41807796,  0.16787228]),
 array([-9.41807796,  0.16787228]),
 array([-9.41807796,  0.16787228]),
 array([-12.3608385 ,   0.22032544]),
 array([-9.90207301e+01,  1.14388460e-12,  8.72124268e-03]),
 array([-9.90207301e+01,  1.14388460e-12,  8.72124268e-03]),
 array([-1.90487717e+12, -9.30842930e+01, -3.03476288e-01,  3.02457459e+02]),
 array([-1.90487717e+12, -9.30842930e+01, -3.03476288e-01,  3.02457459e+02])]

In [ ]:
component_props_level[(component_props_level.time_stamp==pd.Timestamp("2013-01-01 09:00:00")) & (component_props_level.split_number==351)]

In [23]:
vectors_tuple[0][4705]

(numpy.datetime64('2013-01-01T09:00:00.000000000'), (343, 387), 351)

In [29]:
r = split_comp.rocof.values[0]
r = component_props_level.rocof.values[0]
r

-0.1635943574599453

In [61]:
component_props_level.rocof.max() == vectors_tuple[1].max().max()

True

In [20]:
rocof = vectors_tuple[1]

In [36]:
np.unique((rocof[0])), vectors_tuple[0][0]

(array([-0.1141502 ,  1.90212391]),
 (numpy.datetime64('2013-01-01T00:00:00.000000000'), (0, 7), 0))

In [30]:
(rocof == r).sum(axis=1).sum()

0

In [17]:
from utils.config import path_to_indicator_vectors_sclopf

file_name = f"indicator_vector_{indicator_type}_Co2L{co2l}_n{n_nodes}.pklz"
with gzip.open(path_to_indicator_vectors_sclopf + "/" + file_name, "rb") as out:
    vectors_tuple = pickle.load(out)

FileNotFoundError: [Errno 2] No such file or directory: '/srv/data/jlange/power-system-split/no_extensions//results/sclopf/indicator_vectors//indicator_vector_rocof_Co2L0.05_n600.pklz'

In [144]:
np.unique(vectors_tuple[-1][4705])

array([-0.96751748,  0.15415322])

In [124]:
split_properties_lvl.loc[(0.05, "2013-01-01 09:00:00", 351)]

init_failure_0                     343.000000
init_failure_1                     387.000000
lost_load_share_shedding             0.008554
lost_load_share_blackout             0.108986
lost_load_share_total                0.108986
n_components                         2.000000
largest_component_load_share         0.891014
load                            232948.326510
snapshot_weighting                   3.000000
co2l.1                               0.050000
Name: (0.05, 2013-01-01 09:00:00, 351), dtype: float64

In [116]:
component_props_level[component_props_level["total_split_number"]==i]

,time_stamp,init_failure_0,init_failure_1,split_number,rot_energy,power_imbalance,load,rocof,load_share,shedding_load_loss_share,blackout_load_loss_share,total_load_loss_share,snapshot_weighting,total_split_number
26388,2013-01-03 15:00:00,85,802,62,1.257911e+06,645.699027,334573.207780,1.283276e-02,0.997365,0.000000,0.000000,0.000000,3.0,10000
26389,2013-01-03 15:00:00,85,802,62,0.000000e+00,-645.699027,884.095656,-1.614248e+12,0.002635,0.001925,0.002635,0.002635,3.0,10000


In [117]:
split_properties_lvl.iloc[i]

time_stamp                      2013-01-03 15:00:00
split_number                                     62
init_failure_0                                   85
init_failure_1                                802.0
lost_load_share_shedding                   0.001925
lost_load_share_blackout                   0.002635
lost_load_share_total                      0.002635
n_components                                      2
largest_component_load_share               0.997365
load                                  335457.303435
snapshot_weighting                              3.0
co2l.1                                          0.6
Name: 0.6, dtype: object

In [76]:
split_properties.index.get_level_values("split_number")[-10:]

Index([448, 449, 450, 451, 452, 453, 454, 455, 456, 457], dtype='int64', name='split_number')

In [11]:
from utils.config import path_to_vis_results_sclopf

split_properties = pd.read_hdf(save_dir + f"data_filtered_{n_nodes}.h5", index_col=0)

FileNotFoundError: File /srv/data/jlange/power-system-split/no_extensions//results/sclopf/clustering//rocof_blackout_Co2l_n600_lls0.05/data_filtered_600.h5 does not exist

# Plot Maps

## load clustering results

In [3]:
masks_fpath = save_dir + f"/masks_dict.pklz"
with gzip.open(masks_fpath, "rb") as out:
    masks_dict = pickle.load(out)
weights_fpath = save_dir + f"/weights_filtered_dict.pklz"
with gzip.open(weights_fpath, "rb") as out:
    weights_dict = pickle.load(out)

if transformation == None:
    indicator_name = indicator_type
else:
    indicator_name = indicator_type + "_" + transformation


node_cmap = plt.get_cmap('plasma_r')
node_cmap = truncate_colormap(node_cmap, 0.1, 0.9, 1000)
node_cmap.set_under('gainsboro', 1.0)
node_cbar_label="split off main component prob"

edge_cmap = copy.copy(mpl.cm.get_cmap("plasma_r"))
edge_cmap = copy.copy(mpl.cm.get_cmap("cividis_r"))
edge_cmap.set_under('gainsboro', 1.0)


/tmp/ipykernel_709015/1551947974.py:19: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  edge_cmap = copy.copy(mpl.cm.get_cmap("plasma_r"))
/tmp/ipykernel_709015/1551947974.py:20: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  edge_cmap = copy.copy(mpl.cm.get_cmap("cividis_r"))


In [4]:
len(masks_dict[0])

4709980

In [5]:
len(weights_dict[0])

496361

In [6]:
split_properties[split_properties.co2l == 0.0].shape

(496361, 10)

In [ ]:
for clustering_res_path in os.listdir(save_dir)[::-1]:
    if "fitted" in clustering_res_path:
        clustering_res_path = os.path.join(save_dir, clustering_res_path)
        try: 
            with gzip.open(clustering_res_path, "rb") as out:
                clustering_res = pickle.load(out)
            print(clustering_res_path, clustering_res.labels_.shape)
                
            break
        except Exception as e:
            print(f"Error loading {clustering_res_path}: {e}")
            continue
    
        # if clustering_res.labels_.shape[0]==10000:
        #     print(clustering_res_path)

In [ ]:
print(clustering_res_path, clustering_res.labels_.shape)


In [ ]:
clustering_res_path = save_dir + "dbscan_clustering_n600_maxD3_eps0.5_minS50.pklz"
with gzip.open(clustering_res_path, "rb") as out:
    clustering_res = pickle.load(out)

In [7]:
# with gzip.open(clustering_res_path.replace(".pklz", "_labels_all.npy"), "rb") as out:
labels_all = np.load(clustering_res_path.replace("fitted.pklz", "labels_all.npy"), allow_pickle=True)

with gzip.open(clustering_res_path.replace("fitted.pklz", "group_masks.pklz"), "rb") as out:
    group_masks = pickle.load(out)


NameError: name 'clustering_res_path' is not defined

In [8]:
with gzip.open(save_dir+f"unique_blackout_vecs_n{n_nodes}.pklz", "rb") as out:
    unique_blackout_dict = pickle.load(out)
unique_blackout_vecs = np.array(list(unique_blackout_dict.keys()))

In [11]:
unique_blackout_vecs.sum(axis=1).min()

0

In [12]:
with gzip.open(save_dir + f"blackout_vectors_filtered_dict_{n_nodes}.pklz", "rb") as out:
    blackout_vectors_filtered_dict = pickle.load(out)

In [23]:
list(blackout_vectors_filtered_dict.values())[7].sum(axis=1).min()

12

In [25]:
a = np.concatenate(list(blackout_vectors_filtered_dict.values()), axis=0)

In [28]:
a.sum(axis=1).min()

0

In [ ]:
labels_all.shape

In [ ]:
mask_all = np.concatenate(masks, axis=0)
mask_all.shape, labels_all.shape, split_properties.shape

In [ ]:
co2l_list = get_co2_levels(n_nodes)
co2l_list

In [ ]:
split_properties.co2l.unique()

In [ ]:
df = pd.Series(np.array(list(unique_blackout_dict.keys())).sum(axis=1))

In [ ]:
split_properties.loc[split_properties.lost_load_share_blackout.index[:10]].sort_values(by="lost_load_share_blackout", ascending=False).head(10)

In [ ]:
blackout_vectors_filtered 37258

In [ ]:
split_properties.lost_load_share_blackout.hist(bins=np.linspace(0, 1, 100))

In [ ]:
split_properties.iloc[df.sort_values(ascending=False).tail().index]

In [ ]:
plt.hist((np.array(list(unique_blackout_dict.keys())).sum(axis=1)), bins=np.arange(500));

In [ ]:
np.array(list(unique_blackout_dict.keys())).sum(axis=1).max()/unique_blackout_vecs.shape[1]

In [ ]:
split_properties.iloc[:1000].lost_load_share_blackout[masks[0][:1000]]

In [ ]:
labels_filtered = labels_all[mask_all]

In [ ]:
np.unique(labels_filtered)

In [ ]:
labels_all.unique()

In [ ]:
split_properties.shape

In [ ]:
with gzip.open(save_dir+f"unique_blackout_vecs_n{n_nodes}.pklz", "rb") as out:
    unique_blackout_dict = pickle.load(out)
unique_blackout_vecs = np.array(list(unique_blackout_dict.keys()))

In [ ]:
def prepare_clusters_for_analysis(clustering_res):
    unique_idx_to_ids = [val["idxs"] for val in unique_blackout_dict.values()]

    labels = clustering_res.labels_

    # Number of clusters in labels, ignoring noise if present.
    n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise_ = list(labels).count(-1)

    unique_labels = set(labels)
    core_samples_mask = np.zeros_like(labels, dtype=bool)
    core_samples_mask[clustering_res.core_sample_indices_] = True

    class_member_masks = [labels == k for k in unique_labels]
    group_means = np.array([np.mean(unique_blackout_vecs[labels==k], axis=0) for k in unique_labels if k != -1])
    groups = {k: np.concatenate([unique_idx_to_ids[i] for i in range(len(unique_idx_to_ids)) if labels[i]==k]) for k in unique_labels if k != -1}

    all_idxs = list(np.concatenate(list(groups.values())))
    all_idxs.sort()
    labels_all = np.array([None]*split_properties.shape[0])

    for group,idxs in groups.items():
        labels_all[idxs] = group
    np.save(".".join(clustering_res_path.split(".")[:-1])+"_labels_all.npy", labels_all)

    group_masks = {k: np.array(labels_all == k) for k in unique_labels if k != -1}
    with gzip.open(".".join(clustering_res_path.split(".")[:-1])+"group_masks.pklz", "wb") as out:
        pickle.dump(group_masks, out)

# prepare_clusters_for_analysis()

In [ ]:

# co2l_key = "all"
# labels, centroids, samples_per_centroid, centroid_mean_distance, inertia, silhouette_avg = load_clustering(n_nodes, co2l_list, n_clusters, indicator_type, save_dir, transformation=transformation)
plotting_dir = path_to_clustering_results + indicator_name + "/"
plotting_dir = save_dir
# os.makedirs(plotting_dir, exist_ok=True)
failed_edges_indicator_vectors = load_masked_indicator_vectors("failed_edges", masks=masks)
rocof_indicator_vectors = load_masked_indicator_vectors("rocof", masks=masks)

### calculate centroid group properties

In [ ]:
# group_masks = {
#  group: np.logical_or.reduce([labels == label for label in centroid_groups_dict[group]]) for group in centroid_groups_dict.keys()
# }

In [ ]:
split_lost_load = get_lost_load_share(masks=masks, co2l_list=co2l_list, lost_load_type="total", )
split_weighted_lost_load = split_lost_load * weights

centroid_weighted_lost_load = np.array([np.sum(split_weighted_lost_load[labels == i]) for i in range(len(centroids))])
# centroid_weighted_lost_load_by_co2 = np.array([np.sum(split_weighted_lost_load[labels == i]) for i in range(len(centroids))])
centroid_lost_load_share = centroid_weighted_lost_load/centroid_weighted_lost_load.sum()
centroid_groups_lost_load_share = {group_name: np.sum(centroid_lost_load_share[centroid_group]) for group_name,centroid_group in centroid_groups_dict.items()}
centroid_groups_lost_load_share = {key:centroid_groups_lost_load_share[key] for key in sorted(centroid_groups_lost_load_share,key=centroid_groups_lost_load_share.get, reverse=True)}
centroid_groups_dict = {key:centroid_groups_dict[key] for key in centroid_groups_lost_load_share}

masks_sig_to_co2 = []
n_significant_splits = sum(np.concatenate(masks))
start_ind_lvl = 0
end_ind_lvl = 0
for mask in masks:
    end_ind_lvl += sum(mask)
    mask_lvl = np.zeros(n_significant_splits, dtype="bool")
    mask_lvl[start_ind_lvl:end_ind_lvl] = True
    masks_sig_to_co2.append(mask_lvl)
    
    start_ind_lvl += sum(mask)
    
# get masked lost load array
# co2l_list = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8])
co2_lvls_hist = [0.6, 0.2,0.0]
co2l_inds_hist = [np.where(np.array(co2l_list)==co2l)[0][0] for co2l in co2_lvls_hist]
lost_loads = []
for co2l_ind in co2l_inds_hist:
    co2 = co2l_list[co2l_ind]
    lost_loads.append(pd.read_csv(path_to_evaluation_results+f"split_properties_Co2L{co2}_n400.csv", index_col=0).lost_load_total_share.values[masks[co2l_ind]])
group_df = pd.DataFrame.from_dict(centroid_groups_lost_load_share, orient="index", columns=["lost load share"])
for co2_ind, co2 in enumerate(co2l_list):
    centroids_weighted_lost_load_lvl = np.array([np.sum(split_weighted_lost_load[(labels == i) & masks_sig_to_co2[co2_ind]]) for i in range(len(centroids))])
    group_df[f"lost_load_co2l{co2}"] = np.array([np.sum(centroids_weighted_lost_load_lvl[list(centroid_groups_dict.values())[i_group]]) for i_group in range(len(centroid_groups_dict))])
    
centroid_samples_weighted = np.array([sum((labels==i)*weights) for i in range(n_clusters)])
group_weighted_mean_centroids = {key: np.sum(centroids[centroid_groups_dict[key]].transpose()*centroid_samples_weighted[centroid_groups_dict[key]], axis=1)/sum(centroid_samples_weighted[centroid_groups_dict[key]]) for key in centroid_groups_dict.keys()}
edge_centroids = np.array([np.average(failed_edges_indicator_vectors[labels==i], axis=0, weights=weights[labels==i]) for i in range(n_clusters)])
group_weighted_mean_centroids_edges = {key: np.sum((edge_centroids[centroid_groups_dict[key]].transpose()*centroid_samples_weighted[centroid_groups_dict[key]]), axis=1)/sum(centroid_samples_weighted[centroid_groups_dict[key]]) for key in centroid_groups_dict.keys()}
group_weighted_samples = {key: sum(centroid_samples_weighted[centroid_groups_dict[key]]) for key in centroid_groups_dict.keys()}

### plot all clusters sorted by groups

In [ ]:
if transformation == None:
    indicator_name = indicator_type
else:
    indicator_name = indicator_type + "_" + transformation

# if "main" in transformation:
#     node_cbar_label="split off main component prob"
# else:
#     node_cbar_label=None
node_cbar_label = "blackout probability"

custom_order = np.concatenate(list(centroid_groups_dict.values())).astype(int)
group_affiliation = np.array([None for i in range(n_clusters)])
for group, centroid_indices in centroid_groups_dict.items():
    group_affiliation[centroid_indices] = group

print(indicator_name, n_clusters)
os.makedirs(plotting_dir, exist_ok=True)
plot_clusters_lost_load(nx_graph, pos, indicator_name, co2l_list, samples_per_centroid, centroids, labels,cmap=node_cmap, 
                        n_subplots=n_clusters, 
                        centroid_mean_distance=centroid_mean_distance, edge_centroids=edge_centroids, edge_cmap=edge_cmap,
                        cbar_label=node_cbar_label, mask=masks,
                        custom_order=custom_order,
                        ncols=4,
                        group_affiliation=group_affiliation,
                        save_dir=path_to_figures,
                        show_ind = False,
                        show=True,
                        )
plot_clusters_lost_load(nx_graph, pos, indicator_name, co2l_list, samples_per_centroid, centroids, labels,cmap=node_cmap, 
                        n_subplots=n_clusters, 
                        centroid_mean_distance=centroid_mean_distance, edge_centroids=edge_centroids, edge_cmap=edge_cmap,
                        cbar_label=node_cbar_label, mask=masks,
                        custom_order=custom_order,
                        ncols=4,
                        group_affiliation=group_affiliation,
                        save_dir=save_dir,
                        show_ind = True,
                        show=False
                        )

## shedding vs blackout

In [ ]:
# for co2l_ind in co2l_inds_hist:
#     co2 = co2l_list[co2l_ind]
#     # pd.read_csv(path_to_evaluation_results+"split_properties_Co2L{co2}_n400.csv")
#     lost_loads.append(pd.read_csv(path_to_evaluation_results+f"split_properties_Co2L{co2}_n400.csv", index_col=0).lost_load_total_share.values[masks[co2l_ind]])
lost_load_rocof_all, lost_load_shedding_all = [], []
lost_load_rocof, lost_load_shedding = [], []

for co2 in co2l_list:
    split_properties_df = pd.read_csv(path_to_evaluation_results+f"split_properties_Co2L{co2}_n400.csv", index_col=0)
    ll_rocof, ll_shedding = split_properties_df.lost_load_rocof/split_properties_df.total_load, (split_properties_df.lost_load_total - split_properties_df.lost_load_rocof)/split_properties_df.total_load
    lost_load_shedding_all.append(ll_shedding)
    lost_load_rocof_all.append(ll_rocof)
    
    lost_load_shedding.append(ll_shedding.sum())
    lost_load_rocof.append(ll_rocof.sum())

lost_load_rocof_all = pd.concat(lost_load_rocof_all)
lost_load_shedding_all = pd.concat(lost_load_shedding_all)

In [ ]:
plt.plot(co2l_list ,lost_load_rocof, label="rocof")
plt.plot(co2l_list ,lost_load_shedding, label="shedding")
# plt.plot(co2l_list ,lost_load_shedding), label="shedding")
plt.gca().invert_xaxis()
plt.legend()

In [ ]:
plt.plot(co2l_list ,np.array(lost_load_rocof)/np.array(lost_load_shedding), label="rocof")
plt.gca().invert_xaxis()

In [ ]:
plt.scatter(lost_load_rocof_all, lost_load_shedding_all, s=1)
plt.xlabel("lost load rocof")
plt.ylabel("lost load shedding")

In [ ]:
(split_properties_df.n_nodes_split_off/291).hist(log=True)

In [ ]:
plt.scatter(lost_load_rocof_all, lost_load_shedding_all, s=1, 
            # cmap="plasma_r", c=masks[0]
            )
plt.xlabel("lost load rocof")
plt.ylabel("lost load shedding, corrected")
plt.ylim(plt.gca().get_xlim())

In [ ]:
plt.figure(figsize=(4,3))
plt.hist2d(lost_load_rocof_all*100, lost_load_shedding_all*100,
        #    density=True,
           norm="log",
           cmap="plasma_r", bins=(25,25)
            # cmap="plasma_r", c=masks[0]
            )
plt.xlabel("lost load rocof [%]")
plt.ylabel("lost load shedding [%]")
plt.colorbar()
# plt.ylim(0,0.005)
# plt.ylim(plt.gca().get_xlim())
plt.savefig(save_dir + "lost_load_rocof_vs_shedding.pdf", bbox_inches="tight")

In [ ]:
sum(lost_load_rocof_all > 0.98) / num_failures_weighted / 8

In [ ]:
print("the lost load due to ")
lost_load_rocof_all.sum()/lost_load_shedding_all.sum()

In [ ]:
group_df.drop("lost load share", axis=1).T.iloc[::-1,:8]

## group lost load by lvl

In [ ]:
group_df.drop("lost load share", axis=1).T.iloc[::-1,:8].plot.bar(stacked=True, figsize=(10,5), cmap="tab20")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

In [ ]:
group_df.drop("lost load share", axis=1).T.iloc[::-1,:].plot.bar(stacked=True, figsize=(10,5), cmap="tab20")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

In [ ]:
np.arange(60,0,-10)

In [ ]:
lost_load_by_lvl = group_df.drop("lost load share", axis=1).T.iloc[::-1,:8]
lost_load_by_lvl = lost_load_by_lvl.div(lost_load_by_lvl.sum(axis=1), axis=0)*100
# lost_load_by_lvl = lost_load_by_lvl / lost_load_by_lvl.max(axis=)
lost_load_by_lvl.plot()
plt.ylabel("sum lost load share")
plt.xlabel("CO2 level [%]")
plt.gca().set_xticklabels(labels=[""]+list(np.arange(60,-10,-10))+[""])
plt.ylabel("relative lost load share [%]")
plt.savefig(save_dir+"group_lost_load_by_co2_lvl.pdf", bbox_inches="tight")
# plt.gca().set_xticklabels(labels=[f"0.{i}" for i in range(8)])

In [ ]:
lost_load_by_lvl = group_df.drop("lost load share", axis=1).T.iloc[::-1,:8]
# lost_load_by_lvl = lost_load_by_lvl.div(lost_load_by_lvl.sum(axis=1), axis=0)*100
# lost_load_by_lvl = lost_load_by_lvl / lost_load_by_lvl.max(axis=)
lost_load_by_lvl.plot()
plt.ylabel("sum lost load share")
plt.xlabel("CO2 level")
plt.gca().set_xticklabels(labels=[""]+[f"{i}%" for i in list(np.arange(60,-10,-10))]+[""])
plt.ylabel("relative lost load share [%]")
# plt.savefig(save_dir+"group_lost_load_by_co2_lvl.pdf", bbox_inches="tight")
# plt.gca().set_xticklabels(labels=[f"0.{i}" for i in range(8)])

In [ ]:
from labellines import labelLine, labelLines
from itertools import cycle
lines = ["-","--","-.",":","--"]
linecycler = cycle(lines)

lost_load_by_lvl = group_df.drop("lost load share", axis=1).T.iloc[::-1,:7]
# lost_load_by_lvl = lost_load_by_lvl.div(lost_load_by_lvl.sum(axis=1), axis=0)*100
lost_load_by_lvl = lost_load_by_lvl / max(lost_load_by_lvl.max())
# lost_load_by_lvl.plot()
# iterate over the columns
fig, ax = plt.subplots(figsize=(5,4))
for i, (name, y) in enumerate(lost_load_by_lvl.iteritems()):
    ax.plot(co2l, y, label=name, linestyle=next(linecycler))
# ax.plot(lost_load_by_lvl.values)
plt.ylabel("sum lost load share")
plt.xlabel("CO2 level")
plt.gca().set_xticklabels(labels=[""]+[f"{int(C02*100)}%" for C02 in co2l]+[""])
plt.ylabel("lost load [a.u.]")
# place legend to the right of the plot
# fig.legend(loc="center right", bbox_to_anchor=(1.35, 0.5))
# fig.add_axes([])
plt.legend(ncol=2)
# plt.yscale("log")
# plt.xlim(0,0.9)

# labelLines(ax.get_lines(), zorder=2.5)
# labelLines(ax.get_lines(), align=True)
# xvals=[0.5, 0.8, 0.8, 0.80, 1, 0.5, 1, 0.1, 1]
# labelLines(ax.get_lines(), align=False, xvals=xvals)
# plt.savefig(save_dir+"group_lost_load_by_co2_lvl.pdf", bbox_inches="tight")
# plt.gca().set_xticklabels(labels=[f"0.{i}" for i in range(8)])
plt.savefig(save_dir+"group_lost_load_by_co2_lvl.pdf", bbox_inches="tight")

## plot centroid groups

In [ ]:
mpl.style.use('default')
plt.rc('text', usetex=False)
plt.rc('text.latex', preamble=r'\usepackage{amsmath}')

group_weighted_samples_list = list(group_weighted_samples.values())
mean_centroids = np.array(list(group_weighted_mean_centroids.values()))
group_failed_edges=np.array(list(group_weighted_mean_centroids_edges.values()))
original_ind_to_components_ind=None,
mask=None,
custom_order=None,
node_cmap=node_cmap
n_subplots=n_clusters
edge_cmap=edge_cmap
node_cbar_label="split off main component prob"
mask=masks
sum_lost_load_share_per_centroid =np.array(list(centroid_groups_lost_load_share.values()))
group_names = list(centroid_groups_dict.keys())
centroid_inds = list(centroid_groups_dict.values())

if sum_lost_load_share_per_centroid is None:
    sum_lost_load_share_per_centroid = np.array(
        [np.sum(split_lost_load[labels == i]) for i in range(len(mean_centroids))]
    )
    
vmax = 1.0
vmin = 0.001
if "main" in transformation:
    mean_centroids = np.abs(mean_centroids - 1)
co2l_list = np.array(co2l_list)

vmin_edge = 1e-3
vmax_edge = 1

if node_cbar_label is None:
    node_cbar_label = indicator_name.replace("_", " ")

n_subplots = min(n_subplots, len(mean_centroids))
ncols = 3
n_rows = 3
fig_scaling = 4
fig = plt.figure(figsize=(ncols * fig_scaling, n_rows * fig_scaling * 1.3))

gs = GridSpec(2, 1, figure=fig, hspace=0.15, height_ratios=[0.05,n_rows][::-1])
gs_legend_colorax = GridSpecFromSubplotSpec(1, 3, subplot_spec=gs[1, 0])
gs_group = GridSpecFromSubplotSpec(n_rows, 1, subplot_spec=gs[0, 0], hspace=0.2)
plt.rc("text", usetex=False)

plot_count = 0
ind = 0

for row in (range(n_rows)):
    
    gs_row = GridSpecFromSubplotSpec(2, 1, subplot_spec=gs_group[row, 0], hspace=-0.05, height_ratios=[3,1])
    gs_row_map = GridSpecFromSubplotSpec(1, ncols, subplot_spec=gs_row[0, 0], wspace=0.0)
    gs_row_hist = GridSpecFromSubplotSpec(1, ncols, subplot_spec=gs_row[1, 0], wspace=0.18,hspace=0.5)
    
    for column in (range(ncols)):
        
        if plot_count == n_subplots:
            break
        plot_count += 1

        while "ungrouped" in group_names[ind]  or group_names[ind] == "none":
            print(f"skipping {group_names[ind]}")
            ind+=1
        
        controid, samples_in_centriod = mean_centroids[ind], group_weighted_samples_list[ind]
        failed_edges_prob = group_failed_edges[ind]

        ax = fig.add_subplot(gs_row_map[0,column])

        plot_centroid_with_failures(nx_graph, pos, group_failed_edges, node_cmap, edge_cmap, vmax, vmin, vmin_edge, vmax_edge, controid, failed_edges_prob, ax)

        ax.axis("off")

        subtitle = f"={round(sum_lost_load_share_per_centroid[ind]/ sum_lost_load_share_per_centroid.sum()*100, ndigits=1)}%\nn={round(samples_in_centriod/10**6, ndigits=2)}mio"
        subtitle = r"$\bar{R}$" + subtitle
        ax.set_title(
            group_names[ind],
            y=0.90,
            x=-0.05,
            fontsize=14,
            loc="left",
            # horizontalalignment="left",
        )
        ax.text(
            -15,
            52.5,
            subtitle,
            fontsize=14,)
        # sum_lost_load_share_per_centroid.max()
        
        # add gridspec for histograms to gs
        ax_hist = fig.add_subplot(gs_row_hist[0, column])
        plot_group_lost_load_hist_by_co2_single(group_names[ind],centroid_inds[ind],labels,lost_loads,masks_sig_to_co2,co2l_inds_hist,co2_lvls_hist,weights,n_failures_weighted=num_failures_weighted,ax=ax_hist)
        if column!=0:
            ax_hist.set_ylabel("")
        ax_hist.tick_params(axis='y', which='major', pad=0)
        ax_hist.set_xlabel(ax_hist.get_xlabel(), rotation=0, labelpad=0)
        h, l = ax_hist.get_legend_handles_labels()
        ax_hist.legend().set_visible(False)
            
        ind += 1
            
    ax_hist_legend = fig.add_subplot(gs_legend_colorax[0])
    GridSpec(n_rows, 1, figure=fig, hspace=0.2)
    ax_hist_legend.axis("off")
    ax_hist_legend.legend(h, l, loc="center", fontsize=15, ncols=1)

    cbar_ax_node = fig.add_subplot(gs_legend_colorax[1])
    cbar_ax_edge = fig.add_subplot(gs_legend_colorax[2])
    sm_edge = plt.cm.ScalarMappable(
        cmap=edge_cmap, norm=mplcolors.LogNorm(vmin=vmin_edge, vmax=vmax_edge)
    )
    cb_edge = fig.colorbar(sm_edge, cax=cbar_ax_edge, orientation="horizontal")
    cb_edge.ax.tick_params(labelsize=14, width=1.0, which="both")
    cb_edge.ax.set_xlabel(
        "line failure prob", fontsize=14, rotation=0)
    cb_edge.ax.xaxis.set_label_position("top")

    sm_node = plt.cm.ScalarMappable(cmap=node_cmap, norm=mplcolors.LogNorm(vmin=vmin, vmax=vmax))
    cb_node = fig.colorbar(sm_node, cax=cbar_ax_node, orientation="horizontal")
    cb_node.ax.tick_params(labelsize=14, width=1.0, which="both")
    # if "main" in node_cbar_label:
    #     cb_node.ax.set_xticks([vmin, 0.5, 1], labels=[vmin, 0.5, 1])
    cb_node.ax.set_xlabel(node_cbar_label, fontsize=14, rotation=0)
    cb_node.ax.xaxis.set_label_position("top")

fig.savefig(
    f"{path_to_figures}/groups_co2l{co2_lvls_hist}_{ncols}cols_{n_rows}rows.pdf",
    bbox_inches="tight",
)

In [ ]:
mpl.style.use('default')
plt.rc('text', usetex=False)
plt.rc('text.latex', preamble=r'\usepackage{amsmath}')

group_weighted_samples_list = list(group_weighted_samples.values())
mean_centroids = np.array(list(group_weighted_mean_centroids.values()))
group_failed_edges=np.array(list(group_weighted_mean_centroids_edges.values()))
original_ind_to_components_ind=None,
mask=None,
custom_order=None,
node_cmap=node_cmap
n_subplots=n_clusters
edge_cmap=edge_cmap
node_cbar_label="split off main component prob"
mask=masks
sum_lost_load_share_per_centroid =np.array(list(centroid_groups_lost_load_share.values()))
group_names = list(centroid_groups_dict.keys())
centroid_inds = list(centroid_groups_dict.values())

if sum_lost_load_share_per_centroid is None:
    sum_lost_load_share_per_centroid = np.array(
        [np.sum(split_lost_load[labels == i]) for i in range(len(mean_centroids))]
    )
    
vmax = 1.0
vmin = 0.001
if "main" in transformation:
    mean_centroids = np.abs(mean_centroids - 1)
co2l_list = np.array(co2l_list)

vmin_edge = 1e-3
vmax_edge = 1

if node_cbar_label is None:
    node_cbar_label = indicator_name.replace("_", " ")

n_subplots = min(n_subplots, len(mean_centroids))
ncols = 4
n_rows = 3
fig_scaling = 4
fig = plt.figure(figsize=(ncols * fig_scaling, n_rows * fig_scaling * 1.3))

gs = GridSpec(2, 1, figure=fig, hspace=0.15, height_ratios=[0.05,n_rows][::-1])
gs_legend_colorax = GridSpecFromSubplotSpec(1, 3, subplot_spec=gs[1, 0])
gs_group = GridSpecFromSubplotSpec(n_rows, 1, subplot_spec=gs[0, 0], hspace=0.2)
plt.rc("text", usetex=False)

plot_count = 0
ind = 0

for row in (range(n_rows)):
    
    gs_row = GridSpecFromSubplotSpec(2, 1, subplot_spec=gs_group[row, 0], hspace=-0.05, height_ratios=[3,1])
    gs_row_map = GridSpecFromSubplotSpec(1, ncols, subplot_spec=gs_row[0, 0], wspace=0.0)
    gs_row_hist = GridSpecFromSubplotSpec(1, ncols, subplot_spec=gs_row[1, 0], wspace=0.18,hspace=0.5)
    
    for column in (range(ncols)):
        
        if plot_count == n_subplots:
            break
        plot_count += 1

        while "ungrouped" in group_names[ind]  or group_names[ind] == "none":
            print(f"skipping {group_names[ind]}")
            ind+=1
        
        controid, samples_in_centriod = mean_centroids[ind], group_weighted_samples_list[ind]
        failed_edges_prob = group_failed_edges[ind]

        ax = fig.add_subplot(gs_row_map[0,column])

        plot_centroid_with_failures(nx_graph, pos, group_failed_edges, node_cmap, edge_cmap, vmax, vmin, vmin_edge, vmax_edge, controid, failed_edges_prob, ax)

        ax.axis("off")

        subtitle = f"={round(sum_lost_load_share_per_centroid[ind]/ sum_lost_load_share_per_centroid.sum()*100, ndigits=1)}%\nn={round(samples_in_centriod/10**6, ndigits=2)}mio"
        subtitle = r"$\bar{R}$" + subtitle
        ax.set_title(
            group_names[ind],
            y=0.90,
            x=-0.05,
            fontsize=14,
            loc="left",
            # horizontalalignment="left",
        )
        ax.text(
            -15,
            52.5,
            subtitle,
            fontsize=14,)
        # sum_lost_load_share_per_centroid.max()
        
        # add gridspec for histograms to gs
        ax_hist = fig.add_subplot(gs_row_hist[0, column])
        plot_group_lost_load_hist_by_co2_single(group_names[ind],centroid_inds[ind],labels,lost_loads,masks_sig_to_co2,co2l_inds_hist,co2_lvls_hist,weights,n_failures_weighted=num_failures_weighted,ax=ax_hist)
        if column!=0:
            ax_hist.set_ylabel("")
        ax_hist.tick_params(axis='y', which='major', pad=0)
        ax_hist.set_xlabel(ax_hist.get_xlabel(), rotation=0, labelpad=0)
        h, l = ax_hist.get_legend_handles_labels()
        ax_hist.legend().set_visible(False)
            
        ind += 1
            
    ax_hist_legend = fig.add_subplot(gs_legend_colorax[0])
    GridSpec(n_rows, 1, figure=fig, hspace=0.2)
    ax_hist_legend.axis("off")
    ax_hist_legend.legend(h, l, loc="center", fontsize=15, ncols=1)

    cbar_ax_node = fig.add_subplot(gs_legend_colorax[1])
    cbar_ax_edge = fig.add_subplot(gs_legend_colorax[2])
    sm_edge = plt.cm.ScalarMappable(
        cmap=edge_cmap, norm=mplcolors.LogNorm(vmin=vmin_edge, vmax=vmax_edge)
    )
    cb_edge = fig.colorbar(sm_edge, cax=cbar_ax_edge, orientation="horizontal")
    cb_edge.ax.tick_params(labelsize=14, width=1.0, which="both")
    cb_edge.ax.set_xlabel(
        "line failure prob", fontsize=14, rotation=0)
    cb_edge.ax.xaxis.set_label_position("top")

    sm_node = plt.cm.ScalarMappable(cmap=node_cmap, norm=mplcolors.LogNorm(vmin=vmin, vmax=vmax))
    cb_node = fig.colorbar(sm_node, cax=cbar_ax_node, orientation="horizontal")
    cb_node.ax.tick_params(labelsize=14, width=1.0, which="both")
    # if "main" in node_cbar_label:
    #     cb_node.ax.set_xticks([vmin, 0.5, 1], labels=[vmin, 0.5, 1])
    cb_node.ax.set_xlabel(node_cbar_label, fontsize=14, rotation=0)
    cb_node.ax.xaxis.set_label_position("top")

fig.savefig(
    f"{path_to_figures}/groups_co2l{co2_lvls_hist}_{ncols}cols_{n_rows}rows.pdf",
    bbox_inches="tight",
)

### plot all centoids sorted

In [ ]:
# import os
# import gc
# co2l_list = [[0.0,0.1, 0.2, 0.3, 0.4, 0.5, 0.6]]
# # n_clusters_list = [8, 16, 32, 48, 64, 96, 128]
# # n_clusters_list = [10, 15, 20, 35, 50]
# n_clusters_list = [128]

# indicator_type = "lshare",
# transformations = "main_comp_most_frequent_lshare"


# if transformation == None:
#     indicator_name = indicator_type
# else:
#     indicator_name = indicator_type + "_" + transformation


# edge_cmap = copy.copy(mpl.cm.get_cmap("plasma_r"))
# if "not_zero" in transformation:
#     node_cmap = plt.get_cmap('viridis')
#     node_cmap = truncate_colormap(node_cmap, 0.1, 0.9, 1000)
# if indicator_type == "rocof":
#     node_cmap = plt.get_cmap('seismic')
#     node_cmap = truncate_colormap(node_cmap, 0.15, 0.85, 1000)
#     if transformation == "clipped":
#         node_cmap = plt.get_cmap('viridis')
#         node_cmap = truncate_colormap(node_cmap, 0.1, 0.9, 1000)
# elif indicator_type == "lshare":
#     node_cmap = plt.get_cmap('plasma_r')
#     node_cmap = truncate_colormap(node_cmap, 0.1, 0.9, 1000)
#     node_cmap.set_under('gainsboro', 1.0)
#     edge_cmap = copy.copy(mpl.cm.get_cmap("cividis_r"))

# edge_cmap.set_under('gainsboro', 1.0)
# if "main" in transformation:
#     node_cbar_label="split off main component prob"
# else:
#     node_cbar_label=None
    
    
    
# for co2l in co2l_list:
#     if isinstance(co2l, list):
#         co2l_key = "all"
#     else:
#         co2l_key = co2l
#     for n_clusters in n_clusters_list:
        
#         try:
#             labels, centroids, samples_per_centroid, centroid_mean_distance, inertia, silhouette_avg = load_clustering(n_nodes, co2l, n_clusters, indicator_type, path_to_clustering_results, transformation=transformation)
#         except Exception as e:
#             print(e)
#             continue
        
#         print(indicator_name, n_clusters)
#         plotting_dir = path_to_clustering_results + indicator_name + "/"
#         os.makedirs(plotting_dir, exist_ok=True)
#         failed_edges_indicator_vectors = load_masked_indicator_vectors("failed_edges", masks=masks)
#         plot_clusters_lost_load(nx_graph, pos, indicator_name, co2l, samples_per_centroid, centroids, labels,cmap=node_cmap, n_subplots=n_clusters, centroid_mean_distance=centroid_mean_distance, failed_edges=failed_edges_indicator_vectors, edge_cmap=edge_cmap,
#                                 save_dir=path_to_figures,
#                                 cbar_label=node_cbar_label, mask=masks,
#                                 custom_order=custom_order
#                                 )

#         # plt.close("all")
#         # gc.collect()

## plot all centroids unsorted

In [ ]:
import os
import gc
co2l_list = [[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]]
# n_clusters_list = [8, 16, 32, 48, 64, 96, 128]
# n_clusters_list = [10, 15, 20, 35, 50]
n_clusters_list = [128]

indicator_type = "lshare", 
transformations = "main_comp_most_frequent_lshare",


if transformation == None:
    indicator_name = indicator_type
else:
    indicator_name = indicator_type + "_" + transformation


edge_cmap = copy.copy(mpl.cm.get_cmap("plasma_r"))
if "not_zero" in transformation:
    node_cmap = plt.get_cmap('viridis')
    node_cmap = truncate_colormap(node_cmap, 0.1, 0.9, 1000)
if indicator_type == "rocof":
    node_cmap = plt.get_cmap('seismic')
    node_cmap = truncate_colormap(node_cmap, 0.15, 0.85, 1000)
    if transformation == "clipped":
        node_cmap = plt.get_cmap('viridis')
        node_cmap = truncate_colormap(node_cmap, 0.1, 0.9, 1000)
elif indicator_type == "lshare":
    node_cmap = plt.get_cmap('plasma_r')
    node_cmap = truncate_colormap(node_cmap, 0.1, 0.9, 1000)
    node_cmap.set_under('gainsboro', 1.0)
    edge_cmap = copy.copy(mpl.cm.get_cmap("cividis_r"))

edge_cmap.set_under('gainsboro', 1.0)
if "main" in transformation:
    node_cbar_label="split off main component prob"
else:
    node_cbar_label=None
    
    
    
for co2l in co2l_list:
    if isinstance(co2l, list):
        co2l_key = "all"
    else:
        co2l_key = co2l
    for n_clusters in n_clusters_list:
        
        try:
            labels, centroids, samples_per_centroid, centroid_mean_distance, inertia, silhouette_avg = load_clustering(n_nodes, co2l, n_clusters, indicator_type, path_to_clustering_results, transformation=transformation)
        except Exception as e:
            print(e)
            continue
        
        print(indicator_name, n_clusters)
        plotting_dir = path_to_clustering_results + indicator_name + "/"
        os.makedirs(plotting_dir, exist_ok=True)
        # plotting_dir = None
        failed_edges_indicator_vectors = load_masked_indicator_vectors("failed_edges", masks=masks)
        # plot_clusters(indicator_name, co2l, samples_per_centroid, centroids, cmap=cmap, n_subplots=20, save_dir=plotting_dir)
        plot_clusters_lost_load(nx_graph, pos, indicator_name, co2l, samples_per_centroid, centroids, labels,cmap=node_cmap, n_subplots=n_clusters, centroid_mean_distance=centroid_mean_distance, failed_edges=failed_edges_indicator_vectors, edge_cmap=edge_cmap,
                                # save_dir=plotting_dir,
                                cbar_label=node_cbar_label, mask=masks,
                                # original_ind_to_components_ind = original_ind_to_components_ind
                                # , total_lost_load_share =total_lost_load_share
                                )

        # plt.close("all")
        # gc.collect()

In [ ]:
# for co2 in co2l:
co2 = co2l[0]
file_name = f"indicator_vector_{indicator_type}_Co2L{co2}_n{n_nodes}.pklz"
with gzip.open(path_to_indicator_vectors + "/" + file_name, "rb") as out:
    ind_to_ind, indicator_vectors = pickle.load(out)

In [ ]:
with gzip.open(path_to_indicator_vectors + "/" + file_name, "wb") as out:
    pickle.dump([ind_to_ind, indicator_vectors], out)

In [ ]:
index_by_sample_numbers = np.argsort(samples_per_centroid)[::-1]

mean_lost_load_share_per_centroid = np.array([np.mean(split_lost_load[labels==i]) for i in range(len(centroids))])
centroid_weighted_lost_load = np.array([np.sum(split_lost_load[labels==i]) for i in range(len(centroids))])

In [ ]:
plt.plot(np.arange(len(centroids)), mean_lost_load_share_per_centroid)
plt.plot(np.arange(len(centroids)), centroid_weighted_lost_load)
plt.yscale("log")

In [ ]:
centroid_weighted_lost_load.max()

In [ ]:
plt.hist(split_lost_load, bins=100, log=True);

In [ ]:
f, axes = plt.subplots(figsize=(3*len(centroids),4),ncols=len(centroids))
for cluster_ind,ax in zip(range(len(centroids)), axes):
    # f.add_subplot(1, len(centroids), cluster_ind+1)
    ax.hist(split_lost_load[labels==cluster_ind], bins=100, log=False);
    ax.set_title(f"mean lost load share: {round(np.mean(split_lost_load[labels==cluster_ind]), ndigits=2)}\n total lost load share: {round(np.sum(split_lost_load[labels==cluster_ind]), ndigits=0)}")
plt.tight_layout()

## check post split component sizes

In [ ]:
# cost of synthetic inertia:
incremental_cost_min = 0.002
incremental_cost_max = 0.61
cost_max = incremental_cost_max * 80 * 10e9
cost_min = incremental_cost_min * 80 * 10e9

print("cost range: ", cost_min/10e9, "-", cost_max/10e9, "billion euro")
# cost_min/10e9, cost_max/10e9

In [ ]:
co2l = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]
# co2l = [0.1, 0.8]

indicator_type = "lshare"
results = []
for co2 in co2l:
    file_name = f"indicator_vector_{indicator_type}_Co2L{co2}_n{n_nodes}.pklz"
    with gzip.open(path_to_indicator_vectors + "/" + file_name, "rb") as out:
        # results.append(pickle.load(out)[-1][::100,:])
        vector = pickle.load(out)[-1][:10000,:]
    # vector = [np.max(np.unique(vector[i, :])) for i in range(vector.shape[0])]
    print(co2)
    print(vector.shape)
    # min_max = ([np.min(vector, axis=1), np.max(vector,axis=1)])
    results.append(np.max(vector, axis=1))
        
results = np.concatenate(results)

# Plot edge cluster

In [ ]:
n_failing_lines = [sum(cent>0.7) for cent in list(centroids)]
n_cascades_in_cluster = [sum(np.array(labels)==i) for i in range(20)]
max_failure_prob = [max(cent) for cent in list(centroids)]
# plt.hist(n_failing_lines)

In [ ]:
plt.scatter(n_failing_lines, n_cascades_in_cluster, c=max_failure_prob)
plt.xlabel("Number of failing lines (likelihood > 0.7)")
plt.ylabel("Number of cascades in cluster")
plt.colorbar(label="Maximum failure probability")

In [ ]:
plt.scatter(n_failing_lines, n_cascades_in_cluster, c=kmeans.inertia_)
plt.xlabel("Number of failing lines (likelihood > 0.7)")
plt.ylabel("Number of cascades in cluster")
plt.colorbar()

In [ ]:
##### setup figure #####
selected_co2ls = [co2l]

first_n_centroids = centroids
n_rows = int(np.ceil(len(centroids)/3))
fig, axes = plt.subplots(n_rows, 3, figsize=(10, n_rows*5))
# f = plt.figure(figsize=(21, 10))
# gs = GridSpec(6, 4, figure=f)
# axs_clust = [f.add_subplot(gs[:3, i]) for i in range(4)]
# axs2 = [f.add_subplot(gs[3:6, i]) for i in range(4)]
# f.subplots_adjust(hspace=-0.1, wspace=0.0)
# mpl.style.use('default')
plt.rc('text', usetex=False)
# plt.rc('text.latex', preamble=r'\usepackage{amsmath}')

##### setup parameters #####
labels = [r'\textbf{a}', r'\textbf{b}', r'\textbf{c}', r'\textbf{d}']
vmaxvals = []
vminvals = []

#### Primary failures ######

# cmap = copy.copy(mpl.cm.get_cmap("plasma_r"))
# cmap.set_under('gainsboro', 1.0)
node_cmap = copy.copy(mpl.cm.get_cmap("coolwarm"))

# for count, co2l in enumerate(selected_co2ls[::-1]):
for controid,ax in zip(centroids,axes.flatten()):

    level = np.round(co2l, 2)
    
    # Load likelihoods as dictionary and transform into array
    # c_H_p = [edge_likelihoods_primary[level][(u, v)] for u, v in nx_graph.edges()]
    # c_H_p_log = np.array([np.log10(x) if x > 1e-12 else -np.inf for x in c_H_p])

    # print('Min val prim: {:e}'.format(np.amin(np.array(c_H_p)[np.array(c_H_p)>1e-12])))
    # print('Max val prim: {:e}'.format(np.amax(np.array(c_H_p)[np.array(c_H_p)>1e-12])))
    
    vmax = 1e-3
    vmin =  1e-6

    nodes = nx.draw_networkx_nodes(nx_graph,
                                pos=pos,
                                ax=ax,
                                node_color='black',
                                node_size=0)

    edges = nx.draw_networkx_edges(nx_graph,
                                pos=pos,
                                ax=ax,
                                edge_color=controid,
                                width=2.5,
                                edge_cmap=node_cmap,
                                # edge_vmin=np.log10(vmin),
                                # edge_vmax=np.log10(vmax)
                                )

#     axs_clust[count].axis('off')


# cbar_ax = f.add_axes([0.9, 0.57, 0.005, 0.25]) # f.add_axes([0.89, 0.35, 0.005, 0.45])
# sm = plt.cm.ScalarMappable(
#     cmap=cmap, norm=mplcolors.LogNorm(vmin=vmin, vmax=vmax))
# cb = f.colorbar(sm, cax=cbar_ax)
# cb.ax.tick_params(labelsize=26, width=1.0, which='both')
# axs_clust[3].text(0.95, 0.95,
#             r'$\langle p_{\ell}^{\text{p}}\rangle$',
#             fontsize=25,
#             weight='bold',
#             verticalalignment='center',
#             transform=axs_clust[3].transAxes)

# check spain parts high impact splits 60%

In [ ]:
split_properties_all = []
for co2 in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    split_properties_lvl = pd.read_csv(path_to_evaluation_results+f"split_properties_Co2L{co2}_n400.csv", index_col=0)
    split_properties_lvl["co2"] = co2
    split_properties_all.append(split_properties_lvl)
split_properties_all = pd.concat(split_properties_all)


# split_group_affiliation = np.array(["filtered" for i in range(split_properties_all.shape[0])])
split_properties_all["split_group"] = "filtered"

for group in group_masks.keys():
    # print(group)
    # print(split_properties_all[np.concatenate(masks)][group_masks[group]].shape)
    group_inds = np.array(list(range(split_properties_all.shape[0])))[np.concatenate(masks)][group_masks[group]]
    split_properties_all.iloc[group_inds, -1] = group
    # split_group_affiliation[np.concatenate(masks)][group_masks[group]] = group
split_properties_all.to_csv(path_to_evaluation_results+"split_properties_all.csv")

split_properties_all[split_properties_all.co2==0.6].sort_values("lost_load_total_share", ascending=False).head(30)

In [ ]:
split_properties_all = pd.read_csv(path_to_evaluation_results+"split_properties_all.csv", index_col=0)


In [ ]:
import gc
split_properties_60 = split_properties_all[split_properties_all.co2==0.6]
del split_properties_all
gc.collect()

In [ ]:
split_properties_all[split_properties_all.co2==0.6].sort_values("lost_load_total_share", ascending=False).head(30)

In [ ]:
from utils.config import path_to_visualization_results
component_props_level = pd.read_hdf(path_to_visualization_results + f'component_properties_all_n{n_nodes}.h5')
